# A1 — Colab main-training orchestrator

This notebook calls the repository's `fit` implementation; it does **not** define a second training loop. It is for five Fashion-MNIST models × run seeds **69420, 67, 69** on the committed split (seed **36**). It trains and validates only; **no official-test evaluation** happens here.

First, use the GPU batch pilot and optional 64-sample smoke run. These are **not** benchmark results and can run before CNN/RNN/Transformer settings are frozen. Then agree and commit those models' batch sizes and `device: cuda` in `configs/a1/models/`, review checkpoint frequency, restart Colab for a fresh `main` clone, and pass the strict gate before setting `RUN_MAIN = True`.

Main runs and checkpoints are written under your mounted Drive folder. Re-running skips verified completed pairs, resumes one interrupted pair from `last.pt` when possible, and refuses ambiguous/incompatible artifacts. The first main run pins the repository commit in a Drive ledger. Colab GPU availability and runtime lifetime vary; a single session is not guaranteed to finish all 15 runs. Drive I/O can also be slow or fail—inspect any error before retrying.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/TuanKhai1210/Deep-Learning-Benchmark-Suite.git"
CHECKOUT = Path("/content/deep-learning-benchmark-suite")
ARTIFACT_ROOT = Path("/content/drive/MyDrive/dlbench_a1_main")
LEDGER_PATH = ARTIFACT_ROOT / "run_ledger.json"
MAIN_ROOT = ARTIFACT_ROOT / "main"

MODELS = ("linear", "mlp", "cnn", "rnn", "transformer")
SEEDS = (69420, 67, 69)
SELECTED_MODELS = MODELS
SELECTED_SEEDS = SEEDS
REQUIRE_CUDA = True
RUN_MAIN = False  # Change to True only after every preflight check passes.

## 1. Fixed source checkout and dependencies

If the Drive ledger already exists, the checkout is pinned to its original Git commit. No `git pull` occurs mid-experiment. The package's ML extra installs the dependencies declared by this repo; the following cell prints the actual Colab versions.

In [ ]:
import json
import os
import subprocess
import sys

def git(*args: str) -> str:
    return subprocess.run(
        ["git", *args], cwd=CHECKOUT, check=True, text=True, capture_output=True
    ).stdout.strip()

if not CHECKOUT.exists():
    subprocess.run(
        ["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(CHECKOUT)],
        check=True,
    )
if not (CHECKOUT / ".git").is_dir():
    raise RuntimeError(f"Not a Git checkout: {CHECKOUT}")

os.chdir(CHECKOUT)
if git("status", "--porcelain"):
    raise RuntimeError("Source checkout is dirty. Do not train from uncommitted code.")

if LEDGER_PATH.exists():
    previous = json.loads(LEDGER_PATH.read_text(encoding="utf-8"))
    pinned_commit = previous["identity"]["git_commit"]
    if git("rev-parse", "HEAD") != pinned_commit:
        subprocess.run(["git", "checkout", "--detach", pinned_commit], cwd=CHECKOUT, check=True)

SOURCE_COMMIT = git("rev-parse", "HEAD")
print("Source commit:", SOURCE_COMMIT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[ml]"], cwd=CHECKOUT, check=True)
sys.path.insert(0, str(CHECKOUT / "src"))
import dlbench
print("Package source:", dlbench.__file__)

## 2. Hardware and config inspection

Select a GPU runtime (**Runtime → Change runtime type → GPU**). This cell prints unresolved choices but does not block the pilot. The strict gate before main training will stop until CNN/RNN/Transformer settings are committed.

In [ ]:
import torch
import torchvision
from dlbench.common.config import load_config, validate_config

if sys.version_info < (3, 11):
    raise RuntimeError("This repository requires Python 3.11 or newer.")

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__, "Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU runtime before main training.")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CONFIG_PATHS = {name: CHECKOUT / "configs" / "a1" / "models" / f"{name}.py" for name in MODELS}
CONFIGS = {name: load_config(path) for name, path in CONFIG_PATHS.items()}
problems = {}
for name, config in CONFIGS.items():
    issues = validate_config(config, strict=False)
    if REQUIRE_CUDA and config["run"]["device"].lower() != "cuda":
        issues.append("run.device must be cuda for this GPU comparison.")
    frequency = config["training"].get("save_frequency", 1)
    if frequency > 10:
        print(f"WARNING {name}: save_frequency={frequency}; interrupted runs may lack a recent last.pt.")
    print(name, "batch", config["training"]["batch_size"], "device", config["run"]["device"], "issues", issues)
    if issues:
        problems[name] = issues

if problems:
    print("PILOT ONLY — unresolved main-run decisions:", problems)
if git("status", "--porcelain"):
    raise RuntimeError("Checkout became dirty during setup.")

## 3. Data provenance gate

Download Fashion-MNIST if needed, verify the **existing** committed split manifest, and re-measure normalization on its 50,000 training images. This cell never evaluates the official test set.

In [ ]:
from copy import deepcopy
from dlbench.a1.data.dataset import prepare_data
from dlbench.common.artifacts import compute_data_provenance

split_path = CHECKOUT / CONFIGS["linear"]["data"]["split_file"]
if not split_path.is_file():
    raise RuntimeError(f"Committed split manifest is missing: {split_path}")

prepared = prepare_data(deepcopy(CONFIGS["linear"]))
for name, config in CONFIGS.items():
    for key in ("mean", "std"):
        measured = prepared[f"measured_{key}"]
        declared = config["preprocessing"][key]
        if len(measured) != len(declared) or any(abs(a - b) > 1e-6 for a, b in zip(measured, declared)):
            raise RuntimeError(f"{name}: committed {key} differs from train-only measurement.")
    if config["data"]["split_seed"] != 36 or config["budget"]["run_seeds"] != list(SEEDS):
        raise RuntimeError(f"{name}: split/run seeds differ from the agreed plan.")

PROVENANCE = compute_data_provenance(CONFIGS["linear"])
if any(compute_data_provenance(config) != PROVENANCE for config in CONFIGS.values()):
    raise RuntimeError("Models disagree on split or normalization provenance.")
print(prepared)
print("Provenance:", PROVENANCE)

## 4. GPU batch pilot (optional, not an experiment)

Set `RUN_BATCH_PILOT = True` to try a single synthetic forward/backward/optimizer step at the candidate batch size for CNN/RNN/Transformer. This checks basic GPU memory and runtime feasibility; it does **not** select hyperparameters by validation score, test the real loader, or guarantee that a 100-epoch run will fit. If CUDA runs out of memory, restart the runtime and try a smaller batch.

In [ ]:
import time
from dlbench.a1.models.registry import build_model
from dlbench.common.reproducibility import seed_everything

RUN_BATCH_PILOT = False
PILOT_BATCH_SIZE = 64  # Candidate only; commit the agreed value separately.

if RUN_BATCH_PILOT:
    if not torch.cuda.is_available():
        raise RuntimeError("GPU pilot requires a CUDA runtime.")
    for name in ("cnn", "rnn", "transformer"):
        seed_everything(SEEDS[0])
        torch.cuda.empty_cache()
        model = build_model(CONFIGS[name]["model"]).to("cuda").train()
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=CONFIGS[name]["training"]["learning_rate"],
            weight_decay=CONFIGS[name]["training"]["weight_decay"],
        )
        images = torch.randn(PILOT_BATCH_SIZE, 1, 28, 28, device="cuda")
        targets = torch.arange(PILOT_BATCH_SIZE, device="cuda") % 10
        torch.cuda.reset_peak_memory_stats()
        start = time.perf_counter()
        optimizer.zero_grad(set_to_none=True)
        loss = torch.nn.functional.cross_entropy(model(images), targets)
        loss.backward()
        optimizer.step()
        torch.cuda.synchronize()
        print(
            name, "batch", PILOT_BATCH_SIZE, "step_seconds", round(time.perf_counter() - start, 3),
            "peak_allocated_GiB", round(torch.cuda.max_memory_allocated() / 2**30, 3),
            "finite_loss", bool(torch.isfinite(loss)),
        )
        del model, optimizer, images, targets, loss
else:
    print("Batch pilot skipped. Set RUN_BATCH_PILOT=True to test a candidate on this GPU.")

## 5. Optional real-data smoke run

This uses only 64 training and 64 validation examples per model, at most 10 epochs and batch size 16. Unresolved CPU/batch settings are overridden **in memory for smoke only**, while the repository files remain untouched. Smoke artifacts stay in the temporary Colab runtime and are not main results.

In [ ]:
from dlbench.a1.trainer import fit

RUN_SMOKE = False  # Optional real-data check, never a benchmark.
if RUN_SMOKE:
    for name in SELECTED_MODELS:
        config = load_config(CONFIG_PATHS[name])
        config["run"]["seed"] = SEEDS[0]
        config["run"]["device"] = "cuda"
        config["training"]["batch_size"] = 16
        config["run"]["output_root"] = "/content/a1_smoke_runs"
        result = fit(config, smoke=True)
        print("SMOKE ONLY:", name, result.run_dir)
else:
    print("Smoke skipped. Set RUN_SMOKE=True to check training on this Colab runtime.")

## 6. Strict main-run gate and Drive ledger

After the pilot, commit the chosen CNN/RNN/Transformer batch sizes and `device: cuda` on `main`, then start a **fresh Colab runtime** so the clone contains that commit. This cell must reject unresolved configs. It binds future sessions to one Git commit, split and normalization. Use a new Drive folder for a deliberately revised experiment.

In [ ]:
if problems:
    raise RuntimeError("Main training blocked. Commit the unresolved config choices, then start a fresh runtime: " + repr(problems))
if git("status", "--porcelain"):
    raise RuntimeError("Source checkout became dirty before main training.")

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
IDENTITY = {"git_commit": SOURCE_COMMIT, **PROVENANCE, "models": list(MODELS), "seeds": list(SEEDS)}
if LEDGER_PATH.exists():
    ledger = json.loads(LEDGER_PATH.read_text(encoding="utf-8"))
    if ledger.get("identity") != IDENTITY or not isinstance(ledger.get("completed"), dict):
        raise RuntimeError("Drive ledger belongs to a different experiment. Stop and inspect it.")
else:
    if MAIN_ROOT.exists() and any(MAIN_ROOT.iterdir()):
        raise RuntimeError("Main run folders exist without a ledger. Stop to avoid duplicate runs.")
    ledger = {"identity": IDENTITY, "completed": {}}
    with LEDGER_PATH.open("x", encoding="utf-8") as handle:
        json.dump(ledger, handle, indent=2)

print("Drive output:", ARTIFACT_ROOT)
print("Completed pairs:", sorted(ledger["completed"]))

## 7. Main training

Set `RUN_MAIN = True` in the settings cell, then run this cell. It performs up to 15 **sequential** full-budget runs. If a session ends, rerun the notebook: verified finished pairs are skipped; a single partial pair with a valid `last.pt` is passed to the trainer's resume API. Never treat a smoke run as a main run or choose the best seed after seeing validation results.

In [ ]:
import math

def save_ledger() -> None:
    temporary = LEDGER_PATH.with_name(LEDGER_PATH.name + ".tmp")
    temporary.write_text(json.dumps(ledger, indent=2) + "\n", encoding="utf-8")
    os.replace(temporary, LEDGER_PATH)

def inspect_run(run_dir: Path, name: str, seed: int, *, completed: bool) -> None:
    metadata_path = run_dir / "metadata.json"
    if not metadata_path.is_file():
        raise RuntimeError(f"Missing metadata in {run_dir}")
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    expected = {
        "model_name": name,
        "run_seed": seed,
        "run_mode": "main",
        "git_revision": SOURCE_COMMIT,
        "git_dirty": False,
    }
    for key, value in expected.items():
        if metadata.get(key) != value:
            raise RuntimeError(f"{run_dir}: metadata {key} differs from {value!r}")
    if metadata.get("split", {}).get("sha256") != PROVENANCE["split_hash"]:
        raise RuntimeError(f"{run_dir}: split hash mismatch")
    if metadata.get("normalization", {}).get("sha256") != PROVENANCE["statistics_hash"]:
        raise RuntimeError(f"{run_dir}: normalization hash mismatch")
    if completed:
        metrics_path = run_dir / "metrics.json"
        if not metrics_path.is_file():
            raise RuntimeError(f"{run_dir}: completed record lacks metrics.json")
        metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
        for field in ("val_loss", "val_accuracy", "val_macro_f1"):
            if not math.isfinite(float(metrics[field])):
                raise RuntimeError(f"{run_dir}: invalid {field}")

plan = [(name, seed) for name in SELECTED_MODELS for seed in SELECTED_SEEDS]
if not plan or len(plan) != len(set(plan)):
    raise RuntimeError("Run selection is empty or duplicated.")
if not set(SELECTED_MODELS) <= set(MODELS) or not set(SELECTED_SEEDS) <= set(SEEDS):
    raise RuntimeError("Selection contains an unapproved model or seed.")

print("Selected main runs:", plan)
if not RUN_MAIN:
    print("Main training is DISABLED. Review the plan, then set RUN_MAIN=True in settings.")
else:
    if git("status", "--porcelain") or git("rev-parse", "HEAD") != SOURCE_COMMIT:
        raise RuntimeError("Source checkout changed after the provenance gate.")
    for name, seed in plan:
        key = f"{name}:{seed}"
        config = load_config(CONFIG_PATHS[name])
        config["run"]["seed"] = seed
        config["run"]["output_root"] = str(MAIN_ROOT)  # Operational path only; saved in config.json.
        validate_config(config, strict=True)

        if key in ledger["completed"]:
            run_dir = MAIN_ROOT / ledger["completed"][key]
            inspect_run(run_dir, name, seed, completed=True)
            print("SKIP verified:", key, run_dir.name)
            continue

        candidates = sorted(MAIN_ROOT.glob(f"a1_{name}_main_seed{seed}_*")) if MAIN_ROOT.exists() else []
        if len(candidates) > 1:
            raise RuntimeError(f"{key}: multiple main run folders found; resolve manually.")
        if candidates:
            run_dir = candidates[0]
            inspect_run(run_dir, name, seed, completed=False)
            if (run_dir / "metrics.json").is_file():
                print("Recover completed run:", key, run_dir.name)
            elif (run_dir / "last.pt").is_file():
                print("Resume partial run:", key, run_dir.name)
                fit(config, resume_from=run_dir / "last.pt")
            else:
                raise RuntimeError(f"{key}: partial run has no last.pt; inspect before retrying.")
        else:
            print("Start:", key)
            run_dir = fit(config).run_dir

        inspect_run(run_dir, name, seed, completed=True)
        ledger["completed"][key] = run_dir.name
        save_ledger()
        print("SAVED:", key, run_dir.name)

## 8. Validation-only summary

This summarizes saved **validation** metrics. It does not select a seed or access the official test set. Report mean ± sample standard deviation only when all three agreed seeds for a model are present. Hardware/runtime differences across Colab sessions must be disclosed when comparing timing.

In [ ]:
from statistics import mean, stdev

rows = []
for name in MODELS:
    values = []
    for seed in SEEDS:
        key = f"{name}:{seed}"
        if key not in ledger["completed"]:
            continue
        run_dir = MAIN_ROOT / ledger["completed"][key]
        inspect_run(run_dir, name, seed, completed=True)
        metrics = json.loads((run_dir / "metrics.json").read_text(encoding="utf-8"))
        values.append((seed, metrics["val_accuracy"], metrics["val_macro_f1"], run_dir.name))
        rows.append((name, *values[-1]))
    print(name, f"{len(values)}/3 completed")
    if len(values) == 3:
        print("  validation accuracy mean±sd:", mean(x[1] for x in values), stdev(x[1] for x in values))
        print("  validation macro-F1 mean±sd:", mean(x[2] for x in values), stdev(x[2] for x in values))

print("Saved main runs:", len(rows), "/", len(MODELS) * len(SEEDS))
for row in rows:
    print(row)